In [0]:

-- ===========================================================
-- SECTION 3 : WINDOW FUNCTIONS
-- ============================================================
 
-- Q6. Rank customers by total spend within each city
SELECT
    city,
    customer_name,
    total_spend,
    RANK()       OVER (PARTITION BY city ORDER BY total_spend DESC) AS city_rank,
    DENSE_RANK() OVER (PARTITION BY city ORDER BY total_spend DESC) AS city_dense_rank
FROM (
    SELECT
        c.city,
        c.customer_name,
        SUM(o.order_amount) AS total_spend
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.order_status = 'Delivered'
    GROUP BY c.city, c.customer_name
) ranked;
 
-- Q7. Running total of revenue per restaurant over time
SELECT
    r.restaurant_name,
    o.order_date,
    o.order_amount,
    SUM(o.order_amount) OVER (
        PARTITION BY r.restaurant_id
        ORDER BY o.order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_revenue
FROM restaurants r
JOIN orders o ON r.restaurant_id = o.restaurant_id
WHERE o.order_status = 'Delivered'
ORDER BY r.restaurant_name, o.order_date;
 
-- Q8. Each customer's order vs their own average (LAG + comparison)
SELECT
    c.customer_name,
    o.order_date,
    o.order_amount,
    LAG(o.order_amount) OVER (
        PARTITION BY o.customer_id ORDER BY o.order_date
    )                                                           AS prev_order_amount,
    o.order_amount - LAG(o.order_amount) OVER (
        PARTITION BY o.customer_id ORDER BY o.order_date
    )                                                           AS change_vs_prev,
    AVG(o.order_amount) OVER (PARTITION BY o.customer_id)      AS customer_avg_spend
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_status = 'Delivered'
ORDER BY c.customer_name, o.order_date;
 
-- Q9. Top restaurant per city by revenue (ROW_NUMBER)
SELECT city, restaurant_name, total_revenue
FROM (
    SELECT
        r.city,
        r.restaurant_name,
        SUM(o.order_amount) AS total_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY r.city ORDER BY SUM(o.order_amount) DESC
        ) AS rn
    FROM restaurants r
    JOIN orders o ON r.restaurant_id = o.restaurant_id
    WHERE o.order_status = 'Delivered'
    GROUP BY r.city, r.restaurant_id, r.restaurant_name
) t
WHERE rn = 1;
 
-- Q10. Month-over-month revenue growth
SELECT
    order_month,
    monthly_revenue,
    LAG(monthly_revenue) OVER (ORDER BY order_month) AS prev_month_revenue,
    ROUND(
        (monthly_revenue - LAG(monthly_revenue) OVER (ORDER BY order_month))
        * 100.0
        / LAG(monthly_revenue) OVER (ORDER BY order_month), 2
    ) AS mom_growth_pct
FROM (
    SELECT
        DATE_FORMAT(order_date, '%Y-%m') AS order_month,
        SUM(order_amount)                AS monthly_revenue
    FROM orders
    WHERE order_status = 'Delivered'
    GROUP BY order_month
) monthly;